# Nurikabe sampling trajectory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isomorphicdude/gemma/blob/sudoku-sft/gemma/diffusion/hackable_diffusion_adapter/nurikabe_trajectory_video_colab.ipynb)

Load the [Nurikabe LoRA adapter](https://huggingface.co/isometricneko/diffgemma-nurikabe),
fuse it into the base DiffusionGemma model, then record a real denoising run and
inspect the **sampled string with 10 fields per row** and the **rendered grid**
side by side. Every intermediate state comes from the uniform categorical
diffusion sampler. The grid displays the sampled values, including mistakes,
without validity colours, clue repair, target overlays, or digit filtering.

The held-out example and preprocessing come from
[`configs/sft_nurikabe.py`](https://github.com/isomorphicdude/gemma/blob/sudoku-sft/gemma/diffusion/hackable_diffusion_adapter/configs/sft_nurikabe.py), using its
`eval_ds` pipeline on the validation split of
[`isometricneko/nurikabe10x10`](https://huggingface.co/datasets/isometricneko/nurikabe10x10).
This is the eval split used by that config; it is separate from training.
The base checkpoint, adapter, training step, example, exact prompt and sampling
settings are recorded in `manifest.json`.

## Set up the Colab runtime

Use a GPU runtime with at least 80 GB of GPU memory: *Runtime → Change runtime
type → H100, A100 (80 GB) or G4*. These need Colab Pro, Pro+ or pay-as-you-go
compute units. The bfloat16 base weights alone take 47 GiB, so T4, L4 and
40 GB A100 runtimes cannot run the sampler.

On the first run, the next cell clones the
[`sudoku-sft` branch of the fork](https://github.com/isomorphicdude/gemma/tree/sudoku-sft)
to `/content/gemma` and installs it in editable mode with the pinned
`constraints.txt`. JAX uses its CUDA 13 wheels when the driver supports them
(580 or newer), otherwise CUDA 12. Colab's IPython and Jupyter packages are kept
at their installed versions so that its kernel keeps working. The cell then
**restarts the runtime**: run all cells again afterwards, and the cell only
checks the installation.

Outside Colab, start the notebook from a checkout installed as described in
[FORK_SETUP.md](https://github.com/isomorphicdude/gemma/blob/sudoku-sft/FORK_SETUP.md); the cell then installs nothing.

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/isomorphicdude/gemma.git"
REPO_BRANCH = "sudoku-sft"
JAX_VERSION = "0.11.2"  # The version pinned in constraints.txt.
MIN_GPU_GIB = 70  # The bfloat16 base weights alone take 47 GiB.
# Colab's kernel depends on its own IPython/Jupyter stack, so the pinned
# versions of these packages are not installed.
KEEP_COLAB_PACKAGES = (
    "ipython", "ipython-pygments-lexers", "ipywidgets", "widgetsnbextension",
    "jupyterlab-widgets", "jupyter-core", "traitlets", "comm",
    "matplotlib-inline", "prompt-toolkit", "jedi", "parso", "pandas")


def run_command(*command, **kwargs):
  """Run a command and show its combined output in the notebook."""
  print("$", " ".join(map(str, command)), flush=True)
  result = subprocess.run(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, **kwargs)
  print(result.stdout, flush=True)
  result.check_returncode()


def installed_version(package):
  try:
    return importlib.metadata.version(package)
  except importlib.metadata.PackageNotFoundError:
    return None


if IN_COLAB:
  gpu = subprocess.run(
      ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
       "--format=csv,noheader,nounits"], capture_output=True, text=True)
  if gpu.returncode:
    raise RuntimeError("No GPU found. Choose Runtime > Change runtime type > H100, A100 or G4.")
  gpu_name, gpu_mib, driver = (
      field.strip() for field in gpu.stdout.splitlines()[0].split(","))
  print(f"GPU: {gpu_name}, {int(gpu_mib) / 1024:.1f} GiB, driver {driver}")
  if int(gpu_mib) < MIN_GPU_GIB * 1024:
    raise RuntimeError(f"{gpu_name} has less than {MIN_GPU_GIB} GiB of memory. "
                       "Choose an H100, A100 (80 GB) or G4 runtime.")
  PROJECT_DIR = Path("/content/gemma")
  if not (PROJECT_DIR / "pyproject.toml").is_file():
    run_command("git", "clone", "--depth=1", f"--branch={REPO_BRANCH}",
                REPO_URL, PROJECT_DIR)
  if (installed_version("jax") != JAX_VERSION
      or installed_version("hackable-diffusion") is None):
    cuda = "cuda13" if int(driver.split(".")[0]) >= 580 else "cuda12"
    run_command(sys.executable, "-m", "pip", "install", "-q", "uv>=0.11.28")
    excludes = Path("/tmp/keep_colab_packages.txt")
    excludes.write_text("\n".join(KEEP_COLAB_PACKAGES) + "\n")
    # Remove Colab's JAX CUDA plugins; the install adds the one matching JAX.
    plugins = [name for name in ("jax-cuda12-plugin", "jax-cuda12-pjrt",
                                 "jax-cuda13-plugin", "jax-cuda13-pjrt")
               if installed_version(name)]
    if plugins:
      run_command(sys.executable, "-m", "uv", "pip", "uninstall",
                  "--python", sys.executable, *plugins)
    run_command(sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable,
                "-e", PROJECT_DIR, f"jax[{cuda}]=={JAX_VERSION}", "imageio-ffmpeg",
                "-c", PROJECT_DIR / "constraints.txt", "--excludes", excludes)
    print("Restarting the runtime to load the new packages. Run all cells again.",
          flush=True)
    # Killing the kernel at once can drop the output above before it is sent.
    time.sleep(1)
    os.kill(os.getpid(), 9)
else:
  PROJECT_DIR = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                      if (path / "pyproject.toml").is_file()
                      and (path / "gemma" / "diffusion").is_dir()), None)
  if PROJECT_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the gemma checkout.")
print(f"Project: {PROJECT_DIR}; JAX {installed_version('jax')}")

## Settings

The base checkpoint is read directly from Google's public bucket
(`gs://gemma-data/checkpoints/diffusiongemma-26B-A4B-it`, 37.6 GiB), as in the
upstream DiffusionGemma example. Set `DIFFUSIONGEMMA_CHECKPOINT` to use a local
copy instead. The tokenizer is fetched from the same bucket on first use unless
`GEMMA_CACHE_DIR` (default `~/.gemma`) contains
`tokenizer/tokenizer_gemma4.model`.

The adapter bundle (`adapter_config.json` and its Orbax weights) and the
dataset come from the Hugging Face revisions pinned below; the next section
downloads them to `LORA_DIR` and next to `BAGZ_PATH`. Change `EXAMPLE_INDEX` or
the sampling settings below to adjust the trajectory.

In [ ]:
import gc
import json
import time

import numpy as np
from IPython.display import HTML, Video, display

sys.path.insert(0, str(PROJECT_DIR))
os.environ.setdefault("GEMMA_CACHE_DIR", str(Path.home() / ".gemma"))
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# Without preallocation JAX still stops at 75% of GPU memory, which on an 80 GB
# GPU is too little for the weights plus the stored sampling trajectory.
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")

from gemma.diffusion.hackable_diffusion_adapter import puzzle_trajectory as pt
from gemma.diffusion.hackable_diffusion_adapter import lora_checkpoint as lc

TASK = "nurikabe"
# A gs:// path is read in place; a local directory also works.
BASE_CHECKPOINT = os.path.expanduser(os.environ.get(
    "DIFFUSIONGEMMA_CHECKPOINT", "gs://gemma-data/checkpoints/diffusiongemma-26B-A4B-it"))
LORA_REPO = "isometricneko/diffgemma-nurikabe"
LORA_REVISION = "af7ca99a2ba80d768fd91c0994e23f923d9d5f68"
DATASET_REPO = "isometricneko/nurikabe10x10"
DATASET_REVISION = "8ccd4b2c0b80174df8441da9df3a539a9804f866"
LORA_DIR = Path(os.environ.get(
    "PUZZLE_LORA_DIR", PROJECT_DIR / "nurikabe_lora")).expanduser()
BAGZ_PATH = Path(os.environ.get(
    "PUZZLE_BAGZ_PATH", PROJECT_DIR / "gemma" / "diffusion" /
    "hackable_diffusion_adapter" / "data" / TASK / f"{TASK}_eval.bagz")).expanduser()
EXAMPLE_INDEX = int(os.environ.get("PUZZLE_EXAMPLE_INDEX", "0"))
NUM_STEPS = int(os.environ.get("PUZZLE_NUM_STEPS", "64"))
SEED = 42
TEMPERATURE = 0.7
SECONDS_PER_STEP = float(os.environ.get("DIFFUSION_SECONDS_PER_STEP", "0.2"))
TRAJECTORY_DIR = Path(os.environ.get(
    "DIFFUSION_TRAJECTORY_DIR", PROJECT_DIR / "outputs" / f"{TASK}_trajectory"))
OUTPUT_PREFIX = os.environ.get("DIFFUSION_OUTPUT_PREFIX", f"{TASK}_denoising")
TRAJECTORY_DIR.mkdir(parents=True, exist_ok=True)
print(f"Trajectory and exports: {TRAJECTORY_DIR}")

## Download the adapter and evaluation data

Existing files are reused. From the adapter repository, only
`adapter_config.json` and the Orbax weights are fetched. The bundle was written
by `export_adapter` in [`lora_checkpoint.py`](https://github.com/isomorphicdude/gemma/blob/sudoku-sft/gemma/diffusion/hackable_diffusion_adapter/lora_checkpoint.py)
and holds the LoRA tensors and the base parameter layout used in training, but
no base weights.

The dataset's three JSONL splits are converted by the repository's
[`convert_nurikabe.py`](https://github.com/isomorphicdude/gemma/blob/sudoku-sft/gemma/diffusion/hackable_diffusion_adapter/data/nurikabe/convert_nurikabe.py), which
writes `nurikabe_{train,eval,test}.bagz` next to them; this notebook reads only
the eval file.

In [ ]:
from huggingface_hub import snapshot_download

if not (LORA_DIR / "adapter_config.json").is_file():
  snapshot_download(LORA_REPO, revision=LORA_REVISION, local_dir=LORA_DIR,
                    allow_patterns=["adapter_config.json", "lora_params_*"])
if not BAGZ_PATH.is_file():
  snapshot_download(DATASET_REPO, repo_type="dataset", revision=DATASET_REVISION,
                    local_dir=BAGZ_PATH.parent, allow_patterns=["nurikabe_10x10.*.jsonl"])
  # The converter only serializes records, so TensorFlow does not need the GPU.
  run_command(sys.executable, PROJECT_DIR / "gemma" / "diffusion" /
              "hackable_diffusion_adapter" / "data" / TASK / "convert_nurikabe.py",
              f"--input_dir={BAGZ_PATH.parent}", f"--output_dir={BAGZ_PATH.parent}",
              env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))
print(f"Adapter: {LORA_DIR}")
print(f"Held-out data: {BAGZ_PATH}")

## Load the example, base model and LoRA adapter

The config evaluation pipeline supplies the prompt tokens, including BOS, the
task-specific chat template and right padding. Only the dataset location,
a one-record slice, batch size 1 and workers 0 are changed.

Load the base weights and the adapter separately; reading the base weights from
the bucket takes several minutes. The adapter metadata maps the base checkpoint
to the parameter names, shapes and dtypes used in training. `peft.merge_params`
combines the two trees; `lora.fuse_lora_params` adds each LoRA update to its
base weight. Sampling then uses the model without LoRA wrappers.

In [ ]:
import jax
import jax.numpy as jnp
from gemma import diffusion, gm, peft
from hackable_diffusion import hd
from kauldron import konfig
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_ar_state_handler
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_network
from gemma.diffusion.hackable_diffusion_adapter.hd import lora

if not any(device.platform == "gpu" for device in jax.devices()):
  raise RuntimeError("Sampling requires a GPU-enabled JAX environment.")
if "://" not in BASE_CHECKPOINT and not Path(BASE_CHECKPOINT).is_dir():
  raise FileNotFoundError(f"Set BASE_CHECKPOINT to the base weights: {BASE_CHECKPOINT}")

cfg, batch, example = pt.load_example(
    TASK, PROJECT_DIR, example_index=EXAMPLE_INDEX, bagz_path=BAGZ_PATH)
print(f"JAX {jax.__version__}; devices: {jax.devices()}")
print(f"Base checkpoint: {BASE_CHECKPOINT}")
print(f"Held-out data: {example['dataset']}; record {EXAMPLE_INDEX}")

adapter_info, adapter_params = lc.load_adapter(LORA_DIR)
if adapter_info["task"] != TASK:
  raise ValueError(f"Expected {TASK} adapter, found {adapter_info['task']}.")
started = time.monotonic()
base_params = gm.ckpts.load_params(BASE_CHECKPOINT, restore_concurrent_gb=16)
print(f"Loaded the base weights in {time.monotonic() - started:.0f}s.")
base_params = lc.align_base_params(base_params, adapter_info)
fused_params = lora.fuse_lora_params(peft.merge_params(base_params, adapter_params))
network_params = {"gemma_model": fused_params}
network = hd_gemma_network.WrappedDiffusionGemmaNetwork(
    gemma_model=diffusion.DiffusionGemma_26B_A4B())
jax.block_until_ready(network_params)
del base_params, adapter_params, fused_params
gc.collect()
tokenizer = gm.text.Gemma4Tokenizer()
print(f"Fused rank-{adapter_info['rank']} LoRA adapter from step {adapter_info['checkpoint_step']}.")

## Sample and save every denoising state

Use the config's uniform categorical corruption process, a linear schedule,
`DiscreteDDIMStep` with temperature 0.7, and the same self-conditioning as
`sample_ar_steps64` in `eval/ar_eval.py`. There is one 256-token canvas.
`NUM_STEPS=64` means 64 time points: the random start and 63 denoising updates.

`store_trajectory=True` retains logits as well as tokens temporarily, so this
uses more GPU memory than ordinary evaluation. Reduce `NUM_STEPS` if needed.
Only token IDs and decoded strings are saved to disk.

The displayed span starts at canvas position zero and ends at the **final**
canvas's first stop token, fixed across all states. No text is stripped from
inside that span. `tokens.npy` and `canvas_texts.json` retain the entire canvas,
including padding. `texts.json` retains each exact decoded response, so CPU
rendering does not need a tokenizer or model.

In [ ]:
if NUM_STEPS < 2:
  raise ValueError("NUM_STEPS must be at least 2.")
if cfg.aux.num_canvases != 1:
  raise ValueError("This notebook records one canvas; use a one-canvas config.")
CANVAS_LENGTH = int(cfg.aux.canvas_size)
PAD = tokenizer.special_tokens.PAD
stop_tokens = (tokenizer.special_tokens.EOS, tokenizer.special_tokens.END_OF_TURN,
               tokenizer.special_tokens.BEGIN_OF_TOOL_RESPONSE)
process = konfig.resolve(cfg.aux.corruption_process)
canvas_sampler = hd.sampling.DiffusionSampler(
    time_schedule=hd.sampling.UniformTimeSchedule(),
    stepper=hd.sampling.DiscreteDDIMStep(
        corruption_process=process, temperature=TEMPERATURE, logits_dtype=jnp.bfloat16),
    num_steps=NUM_STEPS, store_trajectory=True,
    update_conditioning_fn=hd_gemma_ar_state_handler.PropagateSelfConditioningFn())

inference_fn = hd.inference.FlaxLinenInferenceFn(network=network, params=network_params)

state_handler = hd_gemma_ar_state_handler.GemmaARStateHandler(
    gemma_network=network, gemma_params=network_params["gemma_model"],
    end_tokens=stop_tokens, pad_token=PAD)

prompt_tokens = jnp.asarray(batch["prompt"], dtype=jnp.int32)
prompt_lengths = jnp.sum(prompt_tokens != PAD, axis=1, dtype=jnp.int32)
started = time.monotonic()
sampler_state = state_handler.init_ar_state(
    batch_size=1, conditioning={"prompt_tokens": prompt_tokens, "prompt_lengths": prompt_lengths},
    canvas_length=CANVAS_LENGTH, max_num_canvases=1)
noise_rng, sample_rng = jax.random.split(jax.random.PRNGKey(SEED))
initial_noise = process.sample_from_invariant(
    key=noise_rng, data_spec=jnp.zeros((1, CANVAS_LENGTH, 1), dtype=jnp.int32))
print(f"Sampling {NUM_STEPS - 1} denoising updates...", flush=True)
last_step, trajectory = canvas_sampler(
    inference_fn=inference_fn, rng=sample_rng, initial_noise=initial_noise,
    conditioning=state_handler.create_conditioning_from_state(sampler_state=sampler_state))
trajectory_tokens = np.asarray(trajectory.xt[:, 0, :, 0], dtype=np.int32)
sampling_seconds = time.monotonic() - started
del last_step, trajectory
gc.collect()

np.save(TRAJECTORY_DIR / "prompt_tokens.npy", np.asarray(prompt_tokens))
manifest = pt.save_trajectory(
    TRAJECTORY_DIR, trajectory_tokens, tokenizer,
    dict(example, task=TASK, model="DiffusionGemma 26B-A4B (fused Nurikabe LoRA)",
         checkpoint=str(LORA_DIR / adapter_info["weights"]),
         checkpoint_step=int(adapter_info["checkpoint_step"]),
         base_checkpoint=str(BASE_CHECKPOINT), adapter=str(LORA_DIR),
         adapter_repo=LORA_REPO, adapter_revision=LORA_REVISION,
         adapter_rank=int(adapter_info["rank"]),
         prompt=tokenizer.decode(np.asarray(prompt_tokens[0, :int(prompt_lengths[0])]).tolist()),
         prompt_len=int(cfg.aux.prompt_len), seed=SEED, num_steps=NUM_STEPS,
         temperature=TEMPERATURE, sampling_seconds=round(sampling_seconds, 2),
         corruption="uniform categorical, linear schedule",
         sampler="DiscreteDDIMStep, uniform time schedule, self-conditioning",
         jax=jax.__version__, device=str(jax.devices()[0])),
    stop_tokens=stop_tokens)
print(f"Saved {len(trajectory_tokens)} real canvas states in {sampling_seconds:.1f}s.")
print("Final response, with grid row breaks:\n" + pt.format_raw_grid(manifest["final_text"], pt.TASK_SIZES[TASK]))

## Inspect the saved trajectory (CPU)

All remaining cells read the saved run. To inspect or export it on CPU without
loading weights, run the setup and settings cells and the cells below, with
`TRAJECTORY_DIR` pointing to the saved trajectory.

The raw view inserts a line break after every 10 whitespace-separated fields;
its expandable **Exact decoded response** preserves the original whitespace.
Unexpected words, invalid numbers and extra fields are retained. The grid places
the first 100 fields in row-major order, leaves missing cells empty, and
keeps extra fields in the string view. Nurikabe `#` is dark sea; every other
field is literal text. These views do not run a puzzle verifier or replace a
changed clue with its original value.

In [ ]:
run = pt.PuzzleTrajectory(TRAJECTORY_DIR)
if run.task != TASK:
  raise ValueError(f"Expected {TASK} trajectory, found {run.task}.")
print(f"Checkpoint: {run.info['checkpoint']}")
print(f"Held-out record: {run.info['example_index']} ({run.info['split']} split)")
print(f"States: {len(run.texts)}; updates: {run.updates}")

import matplotlib.pyplot as plt
from IPython import get_ipython
# The eval config loaded with the example selects matplotlib's headless Agg
# backend, which shows no figures; switch back to inline figures.
get_ipython().run_line_magic("matplotlib", "inline")
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, key, label in zip(axes, ("puzzle", "target"), ("Input puzzle", "Reference solution")):
  ax.imshow(pt.render_grid(TASK, run.info[key], side=540))
  ax.set_title(label)
  ax.axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Play/pause or scrub through every sampled state. The HTML is also portable.
interactive_path = TRAJECTORY_DIR / f"{OUTPUT_PREFIX}.html"
interactive = run.interactive_html(seconds_per_step=SECONDS_PER_STEP)
interactive_path.write_text("<!doctype html><meta charset='utf-8'>" + interactive)
display(HTML(interactive))
# If your notebook frontend blocks inline scripts, open the saved HTML in a browser.
print(f"Saved {interactive_path}")

## Snapshot figure

Show the random start, one third, two thirds and final state. Each panel contains
both the row-formatted string and its grid. Save PNG/PDF copies for slides.

In [ ]:
snapshot_states = [round(fraction * run.updates) for fraction in (0, 1/3, 2/3, 1)]
fig, axes = plt.subplots(2, 2, figsize=(20, 11.25))
for ax, state in zip(axes.flat, snapshot_states):
  ax.imshow(run.render(state, size=(960, 540)))
  ax.axis("off")
fig.tight_layout(pad=0.3)
snapshot_path = TRAJECTORY_DIR / f"{OUTPUT_PREFIX}_trajectory.png"
fig.savefig(snapshot_path, dpi=160, facecolor="white")
fig.savefig(snapshot_path.with_suffix(".pdf"), facecolor="white")
plt.show()

plt.close(fig)
print(f"Saved {snapshot_path}")

## Export playback

Export a **1920×1080 H.264 MP4 at 30 fps**, a **960×540 looping GIF**, a final
poster PNG and a portable interactive HTML file. Both string and grid are visible
in every frame, with the true sampling step and field count. Playback holds the
start for one second and the end for three seconds; it is slower than sampling.
`export.json` records the timing and output names. Change `SECONDS_PER_STEP`
and rerun this cell to change the playback speed without sampling again.

In [ ]:
exports = run.export(prefix=OUTPUT_PREFIX, seconds_per_step=SECONDS_PER_STEP)
for kind, path in exports.items():
  print(f"{kind}: {path}")
display(Video(str(exports["mp4"]), embed=True, width=960))

## Download the outputs

Colab deletes the runtime's files when it disconnects. This cell zips the
trajectory directory (saved states, manifest, snapshots and exports) and, in
Colab, downloads the archive through the browser.

In [ ]:
import shutil

archive = shutil.make_archive(str(TRAJECTORY_DIR), "zip", TRAJECTORY_DIR)
print(f"Saved {archive}")
if IN_COLAB:
  from google.colab import files
  files.download(archive)